# TVS Credit: Dynamic Residual Pricing & Lending Strategy Engine

This notebook predicts liquidation value, converts the forecast into an explainable residual-risk score, recommends LTV/pricing/tenure, and stress-tests the portfolio.

**Core control:** information created after seizure or sale is excluded from model inputs to prevent target leakage. Those fields are used only for validation and business analysis.

## 1. Setup

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn openpyxl joblib

In [ ]:
import json, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
print('Setup complete')

## 2. Load and validate the dataset

In [ ]:
FILE = Path('Analytics Case Study Dataset.xlsx')
assert FILE.exists(), f'Move {FILE.name} into {Path.cwd()}'
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()
df.rename(columns={'Traiffic Challan Amount':'Traffic Challan Amount'}, inplace=True)
print('Shape:', df.shape)
display(df.head())

In [ ]:
required = ['Agmt Id','Agmt Date','Asset Model','Asset Fuel Type',
            'Asset Cost At Disbursal','Loan Amount','LTV','Cust Net IRR','Tenure',
            'OS Balance At Liquidation','Target Sold Amount At Liquidation']
missing_required = [c for c in required if c not in df.columns]
assert not missing_required, f'Missing columns: {missing_required}'
quality = pd.DataFrame({
    'Rows':[len(df)], 'Original columns':[len(df.columns)],
    'Missing cells':[int(df.isna().sum().sum())],
    'Duplicate rows':[int(df.duplicated().sum())],
    'Duplicate agreement IDs':[int(df['Agmt Id'].duplicated().sum())]
})
display(quality)

## 3. Dates and business variables

In [ ]:
date_columns = [c for c in ['Agmt Date','Seizure Date','Sold Date'] if c in df.columns]
for c in date_columns:
    df[c] = pd.to_datetime(df[c], errors='coerce')

if {'Agmt Date','Seizure Date'}.issubset(df.columns):
    df['Calculated Asset Age Months'] = (df['Seizure Date']-df['Agmt Date']).dt.days/30.44
if {'Seizure Date','Sold Date'}.issubset(df.columns):
    df['Calculated Yard Months'] = (df['Sold Date']-df['Seizure Date']).dt.days/30.44
if {'Asset Age Months At Seizure','Months Spent In Yard'}.issubset(df.columns):
    df['Estimated Age At Sale'] = df['Asset Age Months At Seizure'] + df['Months Spent In Yard']

safe_asset_cost = df['Asset Cost At Disbursal'].replace(0, np.nan)
safe_os = df['OS Balance At Liquidation'].replace(0, np.nan)
sold = df['Target Sold Amount At Liquidation']
df['Residual Value Ratio'] = sold/safe_asset_cost
df['Recovery Efficiency Ratio'] = sold/safe_os
df['Residual Shortfall'] = (df['OS Balance At Liquidation']-sold).clip(lower=0)
df['Residual Surplus'] = (sold-df['OS Balance At Liquidation']).clip(lower=0)
df['Residual Shortfall Rate'] = (df['Residual Shortfall']/safe_os).replace([np.inf,-np.inf],np.nan)
df['Recovery Status'] = np.where(sold >= df['OS Balance At Liquidation'],'Full Recovery','Shortfall')
df['Value Depreciation'] = df['Asset Cost At Disbursal']-sold
df['Depreciation Rate'] = df['Value Depreciation']/safe_asset_cost

display(df[['Residual Value Ratio','Recovery Efficiency Ratio','Residual Shortfall',
            'Residual Shortfall Rate','Depreciation Rate']].describe().T)

## 4. Portfolio overview and EDA

In [ ]:
portfolio_summary = pd.DataFrame({
 'Metric':['Agreements','Original asset value','Outstanding at liquidation','Sold amount',
           'Average residual-value ratio','Residual shortfall','Full-recovery rate'],
 'Value':[len(df),df['Asset Cost At Disbursal'].sum(),df['OS Balance At Liquidation'].sum(),
          sold.sum(),df['Residual Value Ratio'].mean(),df['Residual Shortfall'].sum(),
          df['Recovery Status'].eq('Full Recovery').mean()]})
display(portfolio_summary)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(17,4))
sns.histplot(df['Residual Value Ratio'].clip(0,1.5), bins=35, ax=axes[0])
axes[0].set_title('Residual-value ratio')
sns.scatterplot(data=df.sample(min(2500,len(df)),random_state=42),
                x='Asset Cost At Disbursal',y='Target Sold Amount At Liquidation',
                alpha=.35,ax=axes[1])
axes[1].set_title('Asset cost vs sold amount')
model_rv=(df.groupby('Asset Model')['Residual Value Ratio'].agg(['mean','count'])
          .query('count >= 30').sort_values('mean').tail(12))
model_rv['mean'].plot.barh(ax=axes[2],title='Best residual retention by model')
plt.tight_layout(); plt.show()

In [ ]:
segment_tables = {}
for c in ['Asset Model','Asset Fuel Type','Cust Region','Pincode Tier','Asset Generalcondition']:
    if c in df.columns:
        segment_tables[c]=(df.groupby(c,dropna=False)
          .agg(Agreements=('Agmt Id','count'),Average_Sold_Value=('Target Sold Amount At Liquidation','mean'),
               Residual_Value_Ratio=('Residual Value Ratio','mean'),Shortfall_Rate=('Residual Shortfall Rate','mean'))
          .sort_values('Agreements',ascending=False))
        print('\n',c); display(segment_tables[c].head(15))

## 5. Leakage-safe feature design

The prediction is an **origination-time decision**. Therefore seizure date, sold date, outstanding balance at liquidation, asset condition after repossession, yard time, challans, and every target-derived variable are excluded. Using them would make back-test performance look strong but would not work for a new loan.

In [ ]:
target = 'Target Sold Amount At Liquidation'
origination_candidates = [
 'Cust Age','Cust Gender','Cust Cibil Score','Cust Employment Type','Cust Net Salary',
 'Coborrower Flag','App Score Risk','Tenure','Cust Net IRR','Cust Branch','Cust Region',
 'Cust State','Pincode Tier','Asset Disc Flag','Asset Alloy Flag','Asset Variant',
 'Asset Model','Asset Fuel Type','Asset Cost At Disbursal','Loan Amount','LTV'
]
features = [c for c in origination_candidates if c in df.columns]
df['Agmt Year'] = df['Agmt Date'].dt.year
df['Agmt Month'] = df['Agmt Date'].dt.month
features += ['Agmt Year','Agmt Month']

model_df = df.dropna(subset=[target,'Agmt Date']).sort_values('Agmt Date').copy()
split = int(len(model_df)*0.80)
train_df, test_df = model_df.iloc[:split].copy(), model_df.iloc[split:].copy()
X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in features if c not in numeric_features]
print('Train:',X_train.shape,'Test:',X_test.shape)
print('Numeric:',numeric_features) 
print('Categorical:',categorical_features)

## 6. Baseline, Ridge and Random Forest

In [ ]:
numeric_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())])
categorical_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                             ('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=10))])
preprocessor = ColumnTransformer([('num',numeric_pipe,numeric_features),('cat',categorical_pipe,categorical_features)])

models = {
 'Baseline': DummyRegressor(strategy='median'),
 'Ridge': Ridge(alpha=10.0),
 'Random Forest': RandomForestRegressor(n_estimators=300,min_samples_leaf=5,
                    max_features=0.75,n_jobs=-1,random_state=RANDOM_STATE)
}

def metrics(y,p):
    nz=np.abs(y)>1e-9
    return {'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**0.5,
            'MAPE (%)':np.mean(np.abs((y[nz]-p[nz])/y[nz]))*100,'R2':r2_score(y,p)}

fitted, rows = {}, []
for name,estimator in models.items():
    pipe=Pipeline([('prep',preprocessor),('model',estimator)])
    pipe.fit(X_train,y_train)
    pred=np.maximum(pipe.predict(X_test),0)
    fitted[name]=pipe
    rows.append({'Model':name,**metrics(y_test,pred)})
results=pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
display(results.style.format({'MAE':'₹{:,.0f}','RMSE':'₹{:,.0f}','MAPE (%)':'{:.2f}','R2':'{:.3f}'}))
best_name=results.iloc[0]['Model']; best_model=fitted[best_name]
test_df['Predicted Sold Amount']=np.maximum(best_model.predict(X_test),0)
print('Selected model:',best_name)

In [ ]:
fig,ax=plt.subplots(figsize=(6,6))
sns.scatterplot(x=y_test,y=test_df['Predicted Sold Amount'],alpha=.35,ax=ax)
lo=min(y_test.min(),test_df['Predicted Sold Amount'].min()); hi=max(y_test.max(),test_df['Predicted Sold Amount'].max())
ax.plot([lo,hi],[lo,hi],'r--'); ax.set(xlabel='Actual sold amount',ylabel='Predicted sold amount',title='Actual vs predicted'); plt.show()

perm=permutation_importance(best_model,X_test,y_test,n_repeats=5,random_state=42,
                            scoring='neg_mean_absolute_error',n_jobs=-1)
importance=(pd.DataFrame({'Feature':features,'Importance':perm.importances_mean})
            .sort_values('Importance',ascending=False))
display(importance.head(15))

## 7. Residual-risk score and lending recommendation engine

In [ ]:
# Origination-time exposure proxy; OS balance at liquidation is deliberately not used.
safe_loan=test_df['Loan Amount'].replace(0,np.nan)
test_df['Predicted Shortfall']=(test_df['Loan Amount']-test_df['Predicted Sold Amount']).clip(lower=0)
test_df['Predicted Shortfall Rate']=(test_df['Predicted Shortfall']/safe_loan).fillna(0).clip(0,1)

# Volatility proxy is transparent and replaceable with live external indices in production.
fuel_risk=test_df['Asset Fuel Type'].astype(str).str.lower().map(
    {'diesel':65,'petrol':50,'cng':35,'electric':60}).fillna(50)
test_df['Market Volatility Score']=fuel_risk
test_df['Residual Risk Score']=(100*(.75*test_df['Predicted Shortfall Rate'])+
                                 .25*test_df['Market Volatility Score']).clip(0,100).round(1)
test_df['Residual Risk Band']=pd.cut(test_df['Residual Risk Score'],[-.01,25,50,75,100],
                                    labels=['Low','Medium','High','Critical'])

ltv_delta=test_df['Residual Risk Band'].map({'Low':2,'Medium':0,'High':-7,'Critical':-15}).astype(float)
rate_delta=test_df['Residual Risk Band'].map({'Low':-.25,'Medium':0,'High':.75,'Critical':1.50}).astype(float)
tenure_cap=test_df['Residual Risk Band'].map({'Low':72,'Medium':60,'High':48,'Critical':36}).astype(float)
test_df['Recommended LTV']=(test_df['LTV']+ltv_delta).clip(50,95).round(1)
test_df['Recommended Pricing']=(test_df['Cust Net IRR']+rate_delta).clip(lower=0).round(2)
test_df['Recommended Tenure']=np.minimum(test_df['Tenure'],tenure_cap).round().astype(int)
test_df['Profitability Score']=(100-(.65*test_df['Residual Risk Score'])+
    .35*(test_df['Recommended Pricing'].rank(pct=True)*100)).clip(0,100).round(1)

def explain(r):
    return (f"{r['Residual Risk Band']} risk: forecast recovery ₹{r['Predicted Sold Amount']:,.0f}; "
            f"expected shortfall ₹{r['Predicted Shortfall']:,.0f}. "
            f"Recommend LTV {r['Recommended LTV']:.1f}%, rate {r['Recommended Pricing']:.2f}% "
            f"and tenure {r['Recommended Tenure']} months under stated policy guardrails.")
test_df['Recommendation Explanation']=test_df.apply(explain,axis=1)
display(test_df[['Agmt Id','Predicted Sold Amount','Residual Risk Score','Residual Risk Band',
                 'Recommended LTV','Recommended Pricing','Recommended Tenure','Recommendation Explanation']].head())

## 8. Scenario simulation and horizon views

In [ ]:
# These are transparent scenario overlays, not separately trained horizon models.
# The dataset has one liquidation observation per agreement, so supervised 12/24/36-month models
# would require repeated historical valuations at those horizons.
scenario_impacts={'Base':0.00,'Moderate stress':-0.10,'Severe stress':-0.22,
                  'EV acceleration (ICE)':-0.12,'Used-price decline':-0.15}
scenario_summary=[]
for scenario,shock in scenario_impacts.items():
    stressed=test_df['Predicted Sold Amount']*(1+shock)
    shortfall=(test_df['Loan Amount']-stressed).clip(lower=0)
    scenario_summary.append({'Scenario':scenario,'Forecast Value':stressed.sum(),
                             'Expected Shortfall':shortfall.sum(),
                             'Shortfall vs Base':None})
scenario_summary=pd.DataFrame(scenario_summary)
base=scenario_summary.loc[scenario_summary['Scenario'].eq('Base'),'Expected Shortfall'].iloc[0]
scenario_summary['Shortfall vs Base']=scenario_summary['Expected Shortfall']-base
display(scenario_summary)

test_df['Base Residual Value']=test_df['Predicted Sold Amount']
test_df['Moderate Stress Value']=test_df['Predicted Sold Amount']*.90
test_df['Severe Stress Value']=test_df['Predicted Sold Amount']*.78
display(test_df[['Agmt Id','Base Residual Value','Moderate Stress Value','Severe Stress Value']].head())

## 9. Validate strategy and export outputs

In [ ]:
comparison=pd.DataFrame({
 'Metric':['Average LTV','Average Pricing','Average Tenure'],
 'Current':[test_df['LTV'].mean(),test_df['Cust Net IRR'].mean(),test_df['Tenure'].mean()],
 'Recommended':[test_df['Recommended LTV'].mean(),test_df['Recommended Pricing'].mean(),test_df['Recommended Tenure'].mean()]})
display(comparison)

strategy_by_risk=(test_df.groupby('Residual Risk Band',observed=True)
 .agg(Agreements=('Agmt Id','count'),Current_LTV=('LTV','mean'),Recommended_LTV=('Recommended LTV','mean'),
      Current_Rate=('Cust Net IRR','mean'),Recommended_Rate=('Recommended Pricing','mean'),
      Current_Tenure=('Tenure','mean'),Recommended_Tenure=('Recommended Tenure','mean'),
      Predicted_Shortfall=('Predicted Shortfall','mean')).reset_index())
display(strategy_by_risk)

In [ ]:
final_columns=['Agmt Id','Asset Model','Asset Fuel Type','Asset Cost At Disbursal','Loan Amount','LTV',
 'Cust Net IRR','Tenure','Predicted Sold Amount','Predicted Shortfall','Residual Risk Score','Residual Risk Band',
 'Market Volatility Score','Recommended LTV','Recommended Pricing','Recommended Tenure','Profitability Score',
 'Base Residual Value','Moderate Stress Value','Severe Stress Value','Recommendation Explanation']
final_output=test_df[[c for c in final_columns if c in test_df.columns]].copy()
final_output.to_csv('TVS_Residual_Lending_Recommendations.csv',index=False)
results.to_csv('TVS_Model_Performance.csv',index=False)
strategy_by_risk.to_csv('TVS_Strategy_By_Risk.csv',index=False)
joblib.dump({'model':best_model,'features':features},'TVS_Residual_Model.joblib')
with open('TVS_Model_Metadata.json','w') as f:
    json.dump({'selected_model':best_name,'features':features,
               'metrics':results.to_dict(orient='records')},f,indent=2)
print('Exports created successfully')

## 10. Interpretation and implementation controls

- Model estimates liquidation value using only origination-time information.
- Risk score combines forecast shortfall and a transparent market-volatility proxy.
- Recommendations are policy rules, not unconstrained ML decisions; caps must be approved by Credit/Risk.
- A champion–challenger pilot should compare approval rate, yield, LGD, recovery, and fairness by segment.
- Retrain after drift in asset mix, used-price indices, EV adoption, or model error.
- Do not call scenario-adjusted values true 12/24/36-month forecasts until horizon-labelled training data exists.
- GenAI may explain recommendations, but it must not change numeric decisions outside approved rules.